# Nykaa Domain Support Agent — Task 6 (Revised): Order Lookup & Escalation Score

This revision aligns Task 6 with the original capstone brief.

The required function is `check_order_status(record_id: str) -> dict`. It returns the order status, order value, and a designed escalation score in the range [0, 1].

The score combines:
- `delayed_shipment` (binary signal)
- normalized recency derived from `days_since_created`

The escalation threshold is calculated from this generated dataset's score distribution using its 80th percentile. The formula and threshold are project design choices and must be documented in the README.

## 1. Imports and load the Task 1 dataset

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

ORDERS_CSV_PATH = Path.cwd() / "orders.csv"

if not ORDERS_CSV_PATH.exists():
    raise FileNotFoundError(
        f"orders.csv not found at {ORDERS_CSV_PATH.resolve()}. "
        "Place it in the current project folder or update ORDERS_CSV_PATH."
    )

orders_df = pd.read_csv(ORDERS_CSV_PATH)

REQUIRED_COLUMNS = {
    "record_id", "category", "status", "order_value_inr",
    "days_since_created", "delayed_shipment"
}
missing = REQUIRED_COLUMNS - set(orders_df.columns)
if missing:
    raise ValueError(f"Missing required columns: {sorted(missing)}")

orders_df["record_id"] = orders_df["record_id"].astype(str).str.strip()
orders_df["status"] = orders_df["status"].astype(str).str.strip()
orders_df["order_value_inr"] = pd.to_numeric(
    orders_df["order_value_inr"], errors="raise"
)
orders_df["days_since_created"] = pd.to_numeric(
    orders_df["days_since_created"], errors="raise"
)
orders_df["delayed_shipment"] = (
    orders_df["delayed_shipment"].astype(str).str.strip().str.lower()
    .map({"true": True, "false": False})
)

if orders_df["record_id"].duplicated().any():
    raise ValueError("record_id values must be unique.")
if orders_df["delayed_shipment"].isna().any():
    raise ValueError("delayed_shipment must contain only True or False.")
if not orders_df["days_since_created"].between(0, 30).all():
    raise ValueError("days_since_created must be between 0 and 30.")

print("Loaded records:", len(orders_df))
display(orders_df.head())


## 2. Define the escalation formula

We normalize order recency to the interval [0, 1]:

`recency_normalized = days_since_created / 30`

Then calculate:

`escalation_score = 0.70 × delayed_shipment + 0.30 × recency_normalized`

Here, `delayed_shipment` is converted to 1 for True and 0 for False.

Interpretation:
- A delayed order receives a substantial score contribution of 0.70.
- Older orders receive a gradually increasing recency contribution, up to 0.30.
- The final score is always between 0 and 1.

These weights are transparent project assumptions, not official Nykaa rules.

In [ ]:
DELAY_WEIGHT = 0.70
RECENCY_WEIGHT = 0.30
MAX_ORDER_AGE_DAYS = 30

assert np.isclose(DELAY_WEIGHT + RECENCY_WEIGHT, 1.0)

def calculate_escalation_score(order):
    delayed_signal = 1.0 if bool(order["delayed_shipment"]) else 0.0
    recency_normalized = (
        float(order["days_since_created"]) / MAX_ORDER_AGE_DAYS
    )

    score = (
        DELAY_WEIGHT * delayed_signal
        + RECENCY_WEIGHT * recency_normalized
    )

    # Protect against floating-point edge cases.
    return round(float(np.clip(score, 0.0, 1.0)), 4)


orders_df["recency_normalized"] = (
    orders_df["days_since_created"] / MAX_ORDER_AGE_DAYS
)
orders_df["escalation_score"] = orders_df.apply(
    calculate_escalation_score, axis=1
)

print("Formula: score = 0.70 × delayed_signal + 0.30 × (days_since_created / 30)")
print("Minimum score:", orders_df["escalation_score"].min())
print("Maximum score:", orders_df["escalation_score"].max())


## 3. Derive the escalation threshold from this dataset

We use the 80th percentile of the composite escalation-score distribution as the review threshold. This makes the threshold data-derived rather than an arbitrary fixed number.

Orders with a score greater than or equal to this threshold are recommended for escalation. Because percentile thresholds can be affected by tied scores, the actual number flagged may not be exactly 20%.

In [ ]:
ESCALATION_THRESHOLD = float(
    orders_df["escalation_score"].quantile(0.80)
)

orders_df["requires_escalation"] = (
    orders_df["escalation_score"] >= ESCALATION_THRESHOLD
)

print(f"80th-percentile escalation threshold: {ESCALATION_THRESHOLD:.4f}")
print(
    "Orders recommended for escalation:",
    int(orders_df["requires_escalation"].sum()),
    "out of",
    len(orders_df)
)
print(
    "Observed flagged percentage:",
    f"{orders_df['requires_escalation'].mean() * 100:.1f}%"
)

print("\nDataset distribution used to justify the threshold:")
display(orders_df["escalation_score"].describe(percentiles=[0.5, 0.8, 0.9]).to_frame())


## 4. Implement the required `check_order_status` tool

This function looks up one order by its record ID and returns the required fields. It also returns the derived score and the escalation recommendation.

For an unknown ID, it returns `found: False` and a clear error message rather than fabricating order details.

In [ ]:
def check_order_status(record_id: str) -> dict:
    """Look up an order and calculate its 0–1 escalation score.

    Args:
        record_id: Unique order identifier, for example 'ORD1005'.

    Returns:
        A dictionary containing the order status, order value,
        escalation score, and escalation recommendation.
    """
    if record_id is None or not str(record_id).strip():
        return {
            "found": False,
            "record_id": record_id,
            "error": "A non-empty record_id is required."
        }

    requested_id = str(record_id).strip().upper()
    matches = orders_df[
        orders_df["record_id"].str.upper() == requested_id
    ]

    if matches.empty:
        return {
            "found": False,
            "record_id": requested_id,
            "error": f"No order found for record_id {requested_id}."
        }

    order = matches.iloc[0].to_dict()
    score = calculate_escalation_score(order)

    return {
        "found": True,
        "record_id": order["record_id"],
        "status": order["status"],
        "order_value_inr": float(order["order_value_inr"]),
        "escalation_score": score,
        "escalation_threshold": round(ESCALATION_THRESHOLD, 4),
        "requires_escalation": score >= ESCALATION_THRESHOLD
    }


# Test a known order and an unknown order.
print("KNOWN ORDER")
print(check_order_status("ORD1005"))

print("\nUNKNOWN ORDER")
print(check_order_status("ORD9999"))


## 5. Validate the function and score range

In [ ]:
# Known ID lookup
known_result = check_order_status("ORD1001")
assert known_result["found"] is True

# Required output fields
for required_key in ["status", "order_value_inr", "escalation_score"]:
    assert required_key in known_result

# Score bounds for every generated order
all_scores = orders_df["escalation_score"]
assert all(all_scores.between(0, 1))

# Delayed shipment must contribute more than a non-delayed order
test_delayed = {
    "delayed_shipment": True,
    "days_since_created": 10
}
test_not_delayed = {
    "delayed_shipment": False,
    "days_since_created": 10
}
assert calculate_escalation_score(test_delayed) > calculate_escalation_score(test_not_delayed)

# Case-insensitive lookup
assert check_order_status("  ord1001  ")["found"] is True

# Unknown ID
assert check_order_status("ORD9999")["found"] is False

print("PASS: check_order_status returns the required fields.")
print("PASS: Every escalation score is within [0, 1].")
print("PASS: Score combines delayed shipment and normalized recency.")
print("PASS: Threshold is derived from the dataset's 80th percentile.")
print("PASS: Known, case-insensitive, and unknown ID tests passed.")


## 6. Inspect the order-level scores and escalation distribution

In [ ]:
display(
    orders_df[
        [
            "record_id", "status", "order_value_inr",
            "days_since_created", "delayed_shipment",
            "recency_normalized", "escalation_score",
            "requires_escalation"
        ]
    ].sort_values("escalation_score", ascending=False).head(15)
)

print("\nEscalation recommendation distribution:")
display(
    orders_df["requires_escalation"]
    .value_counts()
    .rename(index={True: "Recommend escalation", False: "No escalation"})
    .to_frame("number_of_orders")
)


## 7. Save the scored dataset and formula evidence

In [ ]:
OUTPUT_PATH = Path.cwd() / "orders_with_escalation_scores.csv"
orders_df.to_csv(OUTPUT_PATH, index=False)

formula_evidence = {
    "formula": "0.70 * delayed_signal + 0.30 * (days_since_created / 30)",
    "delay_weight": DELAY_WEIGHT,
    "recency_weight": RECENCY_WEIGHT,
    "normalization_max_days": MAX_ORDER_AGE_DAYS,
    "threshold_method": "80th percentile of composite escalation_score in generated dataset",
    "threshold_value": round(ESCALATION_THRESHOLD, 4),
    "number_of_orders": int(len(orders_df)),
    "number_recommended_for_escalation": int(orders_df["requires_escalation"].sum()),
    "flagged_percentage": round(float(orders_df["requires_escalation"].mean() * 100), 2)
}

import json
EVIDENCE_PATH = Path.cwd() / "task6_escalation_formula.json"
EVIDENCE_PATH.write_text(json.dumps(formula_evidence, indent=2), encoding="utf-8")

print("Scored dataset saved to:", OUTPUT_PATH.resolve())
print("Formula evidence saved to:", EVIDENCE_PATH.resolve())
print("\nPASS: Revised Task 6 satisfies the required 0–1 composite-score design.")
